In [1]:
# Install required libraries
!pip install transformers pandas torch qwen_vl_utils

In [2]:
import pandas as pd
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from torch.amp import autocast
import os

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clear the CUDA cache to free up memory
torch.cuda.empty_cache()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Clear the CUDA cache to free up memory
torch.cuda.empty_cache()

# Load the model on the available device(s)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load the default processor for the model
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
import requests
import pandas as pd
import torch
from torch.amp import autocast
from PIL import Image  # <-- Import PIL for image resizing

# GitHub repository URL for raw file access
github_repo_url = "https://raw.githubusercontent.com/Namprire/multimodal/test1/"

# Download the CSV file containing the questions and options
csv_url = github_repo_url + "Validation/validation_without_answers.csv"
csv_file = "validation_without_answers.csv"

response = requests.get(csv_url)
if response.status_code == 200:
    with open(csv_file, "wb") as file:
        file.write(response.content)
else:
    raise Exception(f"Failed to download CSV. Status code: {response.status_code}")

# Read the CSV file
df = pd.read_csv(csv_file)

# Output CSV file
output_csv_file = "test_validation.csv"

# Load existing results to skip already processed images
try:
    existing_df = pd.read_csv(output_csv_file)
    processed_images = set(existing_df["file_name"].tolist())
except FileNotFoundError:
    processed_images = set()

# Get the list of image files from the GitHub repository
image_folder_url = github_repo_url + "Validation/images/"
image_files = df["file_name"].tolist()

# Open CSV file in append mode
with open(output_csv_file, "a") as output_file:
    # Write header if file is new
    if not processed_images:
        output_file.write("file_name,answer\n")

    # Process images one by one
    for image_file in image_files:
        if image_file in processed_images:
            print(f"Skipping {image_file}, already processed.")
            continue  # Skip images that are already processed

        image_url = image_folder_url + image_file
        image_path = image_file

        # 1) Download the image file
        response = requests.get(image_url)
        if response.status_code == 200:
            with open(image_path, "wb") as file:
                file.write(response.content)

            # 2) Resize the image to reduce VRAM usage
            try:
                img = Image.open(image_path).convert("RGB")
                # Example: limit dimensions to max 512×512
                img.thumbnail((512, 512), Image.Resampling.LANCZOS)
                img.save(image_path)  # Overwrite original file with the resized version
            except Exception as e:
                print(f"Could not resize {image_file}: {e}")

        else:
            print(f"Failed to download {image_file}. Skipping...")
            # Write something so we don't retry later
            output_file.write(f"{image_file},DOWNLOAD_ERROR\n")
            output_file.flush()
            continue

        # 3) Get the corresponding row from the CSV file based on the image file name
        row = df[df["file_name"] == image_file].iloc[0]
        question = row["question"]
        options = [row["option1"], row["option2"], row["option3"], row["option4"]]

        # Prepare the options text for the prompt
        options_text = "\n".join([f"{chr(97 + i)}) {option}" for i, option in enumerate(options)])
        prompt = (
            f"This is the Question that you should answer based on the given picture: {question}\n"
            f"Options:\n{options_text}\n"
            "Please choose the correct option (a, b, c, or d) that answers the question correctly. "
            "Give me one of the correct options as the output."
        )

        # Prepare the message for the model, including the image and the prompt
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        # Apply the chat template to the messages and process the vision information
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        # Prepare the inputs for the model
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        try:
            # Free up GPU memory before inference
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

            # Use no_grad + autocast to save memory
            with torch.no_grad(), autocast(device_type="cuda"):
                generated_ids = model.generate(**inputs, max_new_tokens=32)

            # Trim the generated IDs to exclude the prompt part
            generated_ids_trimmed = [
                out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )

            # Find the model's answer among the options
            model_answer = None
            for i, option in enumerate(options, 1):
                if option in output_text[0]:
                    model_answer = i
                    break

            # If no matching option was found, default to "1"
            if model_answer is None:
                model_answer = 1

            # Write result to CSV
            output_file.write(f"{image_file},{model_answer}\n")
            output_file.flush()

            print(f"Processed {image_file} -> Answer: {model_answer}")

        except torch.cuda.OutOfMemoryError:
            print(f"⚠️ OOM on {image_file}. Recording 'OOM' in CSV.")
            # Record OOM so we won't retry this image
            output_file.write(f"{image_file},OOM\n")
            output_file.flush()

            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            continue

        # Free memory after processing
        del inputs
        del generated_ids
        del generated_ids_trimmed
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print(f"✅ Validation results saved to {output_csv_file}")


Skipping dac522bf-a731-4e63-b389-5d1e91b41887.jpg, already processed.
Processed b4b8888b-9dc2-4b57-a677-a4850e75c382.jpg -> Answer: 3
Skipping 3980867c-fa35-4748-882a-192b7ab7616d.jpg, already processed.
Skipping f8582b04-5906-49c5-8c10-bc67c852a20b.jpg, already processed.
Processed 735a75e6-0c86-4c75-823d-28fae3168c53.jpg -> Answer: 1
Processed 8b16b4b9-c610-4b0b-a5e9-7305ec955865.jpg -> Answer: 3
Skipping 836fd561-2499-455b-8393-be559b703d2d.jpg, already processed.
Skipping 0727f5bf-cb0b-471a-8d77-c4102ab95c5e.jpg, already processed.
Skipping 6262c3d3-8d34-4625-9250-8f41dcdc536e.jpg, already processed.
Skipping 71a5ff68-914a-4259-b9d5-dcb5a2e79ba1.jpg, already processed.
Processed 6b68106a-b981-472d-a7af-ab797c9aad90.jpg -> Answer: 3
Skipping 6d01a5ae-5b4b-48ad-a4fb-ba6512b6a43b.jpg, already processed.
Processed 501b0c77-078b-4cc9-9b21-6b22f730e04c.jpg -> Answer: 3
Processed f692cdd6-dcbf-4b81-9886-c39116254fd5.jpg -> Answer: 2
Skipping 143c69d3-c737-4404-b2c1-9cec6273cfbd.jpg, alrea

Now training the model

In [12]:
!pip install -U bitsandbytes


In [13]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)

# 1) Prepare device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) Define the Dataset
class QwenTrainDataset(Dataset):
    def __init__(self, csv_file, image_folder, processor, max_img_size=512):
        self.df = pd.read_csv(csv_file)
        self.image_folder = image_folder
        self.processor = processor
        self.max_img_size = max_img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_file = row["file_name"]
        question = row["question"]
        options = [row["option1"], row["option2"], row["option3"], row["option4"]]
        correct_answer = row["answer"]

        img_path = os.path.join(self.image_folder, image_file)
        img = Image.open(img_path).convert("RGB")
        img.thumbnail((self.max_img_size, self.max_img_size), Image.Resampling.LANCZOS)

        # Build prompt
        options_text = "\n".join([f"{chr(97 + i)}) {opt}" for i, opt in enumerate(options)])
        prompt = (
            f"Question: {question}\n"
            f"Options:\n{options_text}\n"
            f"The correct option is:"
        )
        # Append correct answer
        full_text = prompt + f" {correct_answer}"

        return {
            "image": img,
            "text": full_text
        }

# 3) Instantiate processor & model
quant_config = BitsAndBytesConfig(load_in_4bit=True)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quant_config
).to(device)

model.train()

# 4) Create train dataloader
train_csv_file = "Train/train_with_answers.csv"
train_image_folder = "Train/images"
train_dataset = QwenTrainDataset(
    csv_file=train_csv_file,
    image_folder=train_image_folder,
    processor=processor
)
train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# 5) Define optimizer & LR scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
num_epochs = 1
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_linear_schedule_with_warmup(optimizer, 0, num_training_steps)

# 6) Training loop
global_step = 0
for epoch in range(num_epochs):
    for batch in train_dataloader:
        img = batch["image"][0]
        text = batch["text"][0]

        # Convert to model inputs
        inputs = processor(
            text=[text],
            images=img,
            return_tensors="pt",
            padding=True
        ).to(device)

        labels = inputs["input_ids"].clone()

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        global_step += 1
        if global_step % 10 == 0:
            print(f"Epoch {epoch}, step {global_step}, loss: {loss.item():.4f}")

# 7) Save your fine-tuned model
model.save_pretrained("./my_finetuned_qwen2vl")
processor.save_pretrained("./my_finetuned_qwen2vl")
print("Fine-tuning complete!")


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [6]:
!ls /content/test_validation.csv
from google.colab import files
files.download('/content/test_validation.csv')


/content/test_validation.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>